In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import rule_setup, display_data_doc, collist  # noqa: E402
from util import (  # noqa: E402
    drop_col_few_distinct,
    OrganEntnahmeID,
    drop_duplicate_columns,
    common_translate,
    split_data,
    collapse_col,
    find_redundant_cols,
    fix_redundancies,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Rows from the {term}`DSO` and {term}`ET` are almost all already joined (The not connected rows are removed due to not being in the target population), while no rows from the {term}`IQTIG` are joined. 

In [ ]:
data = pd.read_parquet(input_data)
_ = split_data(
    data, ["donor_et_dso", "donor_et_id_et", "donor_et_iqtig", "transplant_et_id"]
)

For {term}`IQTIG` and {term}`DSO` rows we can only filter on the recipient identifier, while {term}`ET` data can also be filtered based on the transplants.

In [ ]:
data["donor_id"] = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et", "donor_et_iqtig"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
# data["transplant_organ"] = collapse_col(data.loc[:, ["transplant_organ_dso","transplant_organ_et"]], fun=lambda row: None if row.nunique() > 1 else row.iloc[0])
olres = data["donor_id"].drop_duplicates()
assert not data["donor_id"].isna().any(), "Missing donor info"
targetpop = pd.read_parquet(targetpop_data)

data = data[
    (
        (data["donor_id"].isin(targetpop["donor_et_id_et"]))
        & (
            data["transplant_et_id"].isna()
            | data["transplant_et_id"].isin(targetpop["transplant_et_id"])
        )
    )
].copy()
display(
    Markdown(
        f"""The filter process reduced the numbers of donor in the data ({olres.size}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {data["donor_id"].nunique()} in the processed data.
        """
    )
)
data.drop(columns="donor_id", inplace=True)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

The data from the different institutes needs to be connected (see [](general:ic)). The following table lists the different types of rows, whhich occur in the filtered data and which ID combination they use.

In [ ]:
idcols = ["donor_et_dso", "donor_et_id_et", "donor_et_iqtig", "transplant_et_id"]
data = split_data(data, idcols)
assert len(data) == 3, "Not 3 different row types present!?"

As {term}`IQTIG` only describes a few donors, we drop the {term}`IQTIG` data, which is described in the next data overview.

In [ ]:
display_data_doc(data=data["donor_et_iqtig"].reset_index())

We concatenated the remaining data again, as only {term}`DSO` data which is already joined to {term}`ET` rows is present.   

In [ ]:
data = pd.concat(
    [
        data["donor_et_id_et + transplant_et_id"].reset_index(),
        data["donor_et_dso + donor_et_id_et + transplant_et_id"].reset_index(),
    ]
).reset_index(drop=True)

## Domain Steps

For this file the general plan for domain preprocessing was followed (see [](general:ds)).

### Row Filtering

No further filtering was necessary for this file. (see [](general:rf)).

### Unit Conversions

We applied common translations and removed unit specifiers for single unit columns (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

In [ ]:
cols = data.columns[data.columns.to_series().str.contains("_unit")]
assert (data[cols].nunique() != 1).sum() == 0
dropme = cols[data[cols].nunique() == 1].to_list()
display(
    Markdown(
        f"The columns {collist(dropme)} were removed as only a single unit was used."
    )
)
data.drop(columns=dropme, inplace=True)
del dropme

### Consolidating Columns

We consolidated columns that appear for both {term}`ET` and {term}`DSO` (see [](general:crc))

In [ ]:
red = find_redundant_cols(data)
red["donor_et_id_et"] = ["donor_et_dso", "donor_et_id_et"]
fix_redundancies(data, red)

## Intermediate Dataset

In [ ]:
indcols = ["transplant_et_id", "donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols, axis=0)
data = data.set_index(indcols)

In [ ]:
# Another base class might be necessary, see util.py
# describe columns, without checks for now, order is important
ny = ["no", "yes"]


class OrganExtractionKidney(OrganEntnahmeID):
    any_problems: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Problems During The Extraction",
        description="Were there any problems with the extraction?",
        isin=ny,
    )
    arterial_problems: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Arterial Problems",
        description="What problems were during extraction with the artery?",
        isin=["Kein Füllstück", "Schnitt in Arterie", "Läsion der Intima", "Stenose"],
    )
    artery_patch: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Artery Patch",
        description="Was an artery patch applied?",
        isin=ny,
    )
    backtable_flush: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Backtable Flush Quality",
        description="What was the Observed backtable flush quality?",
        isin=["good", "no", "poor"],
    )
    backtable_flush_solution: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Backtable Flush Solution",
        description="What solution was used for the backtable flush?",
        isin=["Custodiol (HTK)", "andere Perfusionslösung"],
    )
    backtable_flush_solution_ml: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Volume of Backtable Flush Solution",
        description="How much of the solution was used for backtable flush in ml?",
        gt=0,
    )
    biopsy: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Biopsy",
        description="Was an biopsy performed?",
        isin=ny,
    )
    cold_ischemia_time_min: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cold Ischemia Time",
        description="What was the cold ischemia time in min?",
        ge=0,
    )
    comment: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Arteriosclerosis Comment",
        description="What was the comment on the state of the organ? Mainly arteriosclerosis comments.",
    )
    crossclamp_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Crossclamp Date",
        description="When was the crossclamp applied?",
    )
    enbloc: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Enbloc Extraction",
        description="Was the operation performed enbloc?",
        isin=ny,
    )
    eval_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Evaluation Date",
        description="When was the operation evaluated?",
    )
    eval_quality_organ: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Evaluation Organ Quality",
        description="What was the observed quality of the organ after the operation?",
        isin=["good", "acceptable", "average"],
    )
    heparin_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Heparin Date",
        description="When was heparin provided",
    )
    heparin_iu: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Heparin Amount",
        description="Hoch much heparin was provided in IU?",
    )
    nephrectomy_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Nephrectomy Date",
        description="When was the nephrectomy performed?",
    )
    number_arteries: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Number of Arteries",
        description="How many arteries were available?",
        ge=1,
    )
    number_bags: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Number of bags",
        description="How many bags were used?",
        ge=1,
    )
    number_ureter: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Number of Ureter",
        description="How many ureteries were available?",
        ge=0,
    )
    number_vein: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Number of Veins",
        description="How many veins were available?",
        ge=1,
    )
    organ_quality: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Organ Quality",
        description="What was the quality of the organ observed?",
        isin=["good", "acceptable", "poor"],
    )
    perfusion_automatic: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Automatic Perfusion",
        description="Was automatic perfusion used?",
        isin=ny,
    )
    perfusion_coldstart_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Perfusion Date",
        description="When was the perfusion performed?",
    )
    perfusion_quality: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Perfusion Quality",
        description="What was the quality of the perfusion observed",
        isin=["good", "acceptable", "poor"],
    )
    perfusion_solution: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Perfusion Solution",
        description="Which solution was used for perfusion?",
        isin=[
            "HTK/Bretschneider",
            "UW",
            "Celsior",
            "Modified UW",
            "Other",
            "Eurocollins",
            "Marshall",
            "IGL-1",
        ],
    )
    perfusion_volume_ml: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Volume of perfusion Solution",
        description="How much of the solution was used for the perfusion in ml?",
        gt=0,
    )
    quality_assurance_organ_fozen: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="QA frozen organ",
        description="Was the organ frozen?",
        isin=["yes"],
    )
    quality_assurance_packaging_leaking: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="QA package leaking",
        description="Was the package leaking?",
        isin=["yes"],
    )
    quality_assurance_packaging_level: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="QA package fluid level",
        description="Were there problems with the liquid level?",
        isin=["yes"],
    )
    quality_parenchyma: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Quality of the parenchyma",
        description="What was the observed quality of the parenchyma?",
        isin=["Entkapselt", "Tumor", "Partiell entkapselt", "Narben"],
    )
    reason_unusable: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Organ unusability",
        description="What was the reason for the unusability of the organ?",
        isin=[
            "Medical Problems",
            "unknown",
            "Recipients - None",
            "Reason known by DSO",
        ],
    )
    removal_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Removal Date",
        description="When was the organ extracted?",
    )
    reperfusion_color: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Reperfusion Color",
        description="What was the color of the perfusion solution exiting the organ?",
        isin=["Homogen", "Marmoriert", "Dunkelblau"],
    )
    stitches_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Stitches Date",
        description="When were the cuts stichted?",
    )
    transplant_organ: Series[str] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Removed Organ",
        description="Which kidney was extracted?",
        isin=["RKi", "LKi"],
    )
    ureter_length: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Ureter Length",
        description="Was the ureter long or short?",
        isin=["Long", "Short"],
    )
    ureter_problems: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Ureter Problems",
        description="Were there problems with the ureter?",
        isin=["Zu kurz", "Devaskularisiert", "Schnitt im Ureter"],
    )
    urine_production: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Urin Production",
        description="Was there urine production?",
        isin=["good", "no", "average"],
    )
    vein_patch: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Vein Patch",
        description="Was an vein patch applied?",
        isin=ny,
    )
    vein_problems: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Vein Problems",
        description="What problems were during extraction with the vein?",
        isin=["Schnitt in Vene", "Zu kurz"],
    )
    warm_ischemia_time_min: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Warm Ischemia Time",
        description="What was the warm ischemia time in min?",
        ge=0,
    )
    warm_ischemia_time_second_min: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Second Warm Ischemia Time",
        description="What was the second warm ischemia time in min?",
        ge=0,
    )

    class Config:
        title = "Kidney Extraction Dataset"
        description = "Each row represents a single kidney extraction operation. The data is based on the 'element_organ_entnahme_niere.csv' file. It contains data from the ET, DSO and IQTIG."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(OrganExtractionKidney, data)

In [ ]:
OrganExtractionKidney.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    OrganExtractionKidney.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)